# 🚖 下車地址推薦系統

**架構：兩階段推薦**
- Stage 1 (Candidate Generation)：從 `address_v2_suggestion` 撈出用戶歷史下車地址候選集
- Stage 2 (Ranking)：用 **LightGBM LambdaRank** 對候選集排序，輸出 Top-K 推薦

**特徵（12 個）：** 原始頻率 × 6 + log1p 轉換版 × 6

| 特徵 | 說明 |
|---|---|
| `user_end_freq` | 用戶歷史去過該地址幾次（最強信號）|
| `global_end_freq` | 全局熱門度（cold-start 用）|
| `hour_end_freq` | 同時段去該地址的條件頻率 |
| `holiday_end_freq` | 同假日/平日狀態的條件頻率 |
| `dow_end_freq` | 同星期幾的條件頻率 |
| `start_end_freq` | 從同一上車區域出發去該地址的頻率 |

**評估指標：** Recall@K、MRR@K、NDCG@K（K = 1, 3, 5）

---
### 使用說明
1. 把 `address_v2_training_data.parquet` 和 `address_v2_suggestion.parquet` 上傳到 Google Drive
2. 在 **⚙️ 設定** 那格填入正確的檔案路徑
3. 依序執行所有 Cell（`執行階段 → 全部執行`）

## 0｜安裝套件

In [ ]:
# lightgbm --upgrade 確保版本支援 GPU
!pip install lightgbm --upgrade tqdm pyarrow -q

## 1｜掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive 已掛載')

## ⚙️ 設定（請修改這格）

In [ ]:
from pathlib import Path

# ── 資料路徑（請改成你放在 Drive 的實際路徑）──────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive')  # Drive 根目錄，通常不用改
DATA_FOLDER   = DRIVE_ROOT / 'LineGO_data'      # ← 改成你放資料的資料夾

TRAIN_PARQUET = DATA_FOLDER / 'address_v2_training_data.parquet'
SUGG_PARQUET  = DATA_FOLDER / 'address_v2_suggestion.parquet'

# ── Checkpoint 輸出位置（存在 Drive 才不會因 runtime 重啟而遺失）────────
OUTPUT_DIR = DRIVE_ROOT / 'LineGO_checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 模型超參數 ────────────────────────────────────────────────────────
TOP_K_LIST  = [1, 3, 5]
NEG_RATIO   = 4      # 每個正樣本配幾個負樣本
MIN_TRIPS   = 20     # 用戶至少幾筆行程才納入（確保歷史足夠豐富）
N_ROUNDS    = 500    # LightGBM 最大迭代數（有 early stopping 實際可能更少）
EARLY_STOP  = 30     # val NDCG 連續幾輪不進步就停止
SEED        = 42

# ── 時序切分比例（依 created_at 排序後切分，避免 data leakage）──────────
# 切分依「全量行程的筆數」決定，而非依用戶
# 這樣 train / val / test 的時間範圍是嚴格不重疊的
TRAIN_RATIO = 0.75   # 前 75% 筆（約 2026/01 ~ 04）
VAL_RATIO   = 0.10   # 接著 10% 筆（約 2026/04 底）
# 剩餘 15%（約 2026/05 初 ~ 05/17）自動作為 test

FEAT_COLS = ['user_end_freq', 'global_end_freq', 'hour_end_freq',
             'holiday_end_freq', 'dow_end_freq', 'start_end_freq']
ALL_FEAT  = FEAT_COLS + [f'log_{c}' for c in FEAT_COLS]

# 確認檔案存在
for p in [TRAIN_PARQUET, SUGG_PARQUET]:
    status = '✓' if p.exists() else '✗ 找不到'
    print(f'  {status}  {p}')

## 2｜Import

In [ ]:
import gc, json, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from tqdm.notebook import tqdm   # ← notebook 版進度條，有彩色動態效果

print('✓ 所有套件載入完成')

## 2.5｜GPU 偵測

In [ ]:
import subprocess

def detect_gpu():
    """偵測是否有可用的 GPU，並設定 LightGBM 訓練裝置。"""
    global USE_GPU, LGB_DEVICE_PARAMS

    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
             '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0 and result.stdout.strip():
            gpu_info = result.stdout.strip().split('\n')
            print('🟢 GPU 可用！')
            for i, info in enumerate(gpu_info):
                name, mem, driver = [x.strip() for x in info.split(',')]
                print(f'   GPU {i}: {name} | 顯存: {mem} | Driver: {driver}')
            USE_GPU = True
            LGB_DEVICE_PARAMS = {
                'device':          'gpu',
                'gpu_platform_id': 0,
                'gpu_device_id':   0,
            }
            print('   ✓ LightGBM 將使用 GPU 訓練')
        else:
            raise RuntimeError('nvidia-smi 無回應')
    except Exception as e:
        print(f'🔴 未偵測到 GPU（{e}）')
        print('   → 使用 CPU 訓練（可至 執行階段 → 變更執行階段類型 → T4 GPU 切換）')
        USE_GPU = False
        LGB_DEVICE_PARAMS = {}  # 空 dict = CPU 模式，不改任何參數

    # 額外驗證：讓 LightGBM 實際試跑一個最小 GPU dataset
    if USE_GPU:
        try:
            import lightgbm as lgb, numpy as np
            _X = np.random.rand(100, 4).astype(np.float32)
            _y = np.random.randint(0, 2, 100).astype(np.float32)
            _ds = lgb.Dataset(_X, label=_y)
            lgb.train(
                {'objective': 'binary', 'verbosity': -1, **LGB_DEVICE_PARAMS},
                _ds, num_boost_round=3
            )
            print('   ✓ LightGBM GPU 驗證通過')
        except Exception as e:
            print(f'   ⚠️  LightGBM GPU 驗證失敗：{e}')
            print('   → 自動退回 CPU 模式')
            USE_GPU = False
            LGB_DEVICE_PARAMS = {}

USE_GPU = False
LGB_DEVICE_PARAMS = {}
detect_gpu()

## 3｜讀取資料

In [ ]:
def load_data():
    """
    讀取資料並做時序切分：
      1. 全量行程依 created_at 排序
      2. 依筆數切成 train 75% / val 10% / test 15%
         → train 永遠比 val/test 更早，避免 data leakage
      3. lookup 特徵只用 train 行程計算
         → 防止 val/test 期間的行程頻率洩漏進特徵
    """
    steps = ['讀取行程資料', '讀取建議地址資料', '時序切分']
    with tqdm(total=len(steps), desc='[1/5] 讀取資料', unit='step') as pbar:

        pbar.set_postfix_str(steps[0])
        df_trip = pd.read_parquet(TRAIN_PARQUET)
        df_trip['created_at'] = pd.to_datetime(df_trip['created_at'], utc=True)
        # 過濾行程數 >= MIN_TRIPS 的用戶
        cnt   = df_trip.groupby('uid_hash').size()
        valid = cnt[cnt >= MIN_TRIPS].index
        df_trip = df_trip[df_trip['uid_hash'].isin(valid)].copy()
        pbar.update(1)

        pbar.set_postfix_str(steps[1])
        table   = pq.read_table(SUGG_PARQUET)
        df_sugg = pd.DataFrame({c: table.column(c).to_pylist() for c in table.column_names})
        del table; gc.collect()
        pbar.update(1)

        pbar.set_postfix_str(steps[2])
        # 依 created_at 排序後，以筆數做嚴格時序切分
        df_trip = df_trip.sort_values('created_at').reset_index(drop=True)
        n       = len(df_trip)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)

        df_train_raw = df_trip.iloc[:n_train].copy()
        df_val_raw   = df_trip.iloc[n_train : n_train + n_val].copy()
        df_test_raw  = df_trip.iloc[n_train + n_val :].copy()
        pbar.update(1)

    # 印出切分統計
    def _info(name, df):
        t0 = df['created_at'].min().strftime('%Y/%m/%d')
        t1 = df['created_at'].max().strftime('%Y/%m/%d')
        print(f'  {name:<8} {len(df):>7,} 筆  {df["uid_hash"].nunique():>5,} 用戶  '
              f'時間範圍: {t0} ~ {t1}')

    print(f'  ✓ 有效用戶: {df_trip["uid_hash"].nunique():,}，總行程: {n:,}')
    _info('train',   df_train_raw)
    _info('val',     df_val_raw)
    _info('test',    df_test_raw)

    return df_train_raw, df_val_raw, df_test_raw, df_sugg

df_train_raw, df_val_raw, df_test_raw, df_sugg = load_data()

## 4｜預計算頻率查找表

In [ ]:
def build_lookup_tables(df_train_raw: pd.DataFrame) -> dict:
    """
    只用 train 行程計算頻率查找表。
    若使用全量資料，val/test 期間的行程次數會洩漏進特徵，
    導致評估分數過度樂觀（data leakage）。
    """
    lookup_defs = [
        ('user_end',    ['uid_hash',     'end_latlng']),
        ('global_end',  ['end_latlng']),
        ('hour_end',    ['hour_type',    'end_latlng']),
        ('holiday_end', ['is_holiday',   'end_latlng']),
        ('dow_end',     ['dayofweek',    'end_latlng']),
        ('start_end',   ['start_latlng', 'end_latlng']),
    ]
    lookups = {}
    with tqdm(lookup_defs, desc='[2/5] 建立頻率查找表 (僅用 train)', unit='table') as pbar:
        for name, keys in pbar:
            pbar.set_postfix_str(name)
            lookups[name] = df_train_raw.groupby(keys).size().to_dict()

    print(f'  ✓ lookup sizes: {", ".join(f"{k}={len(v):,}" for k,v in lookups.items())}')
    return lookups

lookups = build_lookup_tables(df_train_raw)

## 5｜建立訓練樣本

In [ ]:
def _get_feat(lookups, uid, end, hour, holiday, dow, start):
    f = [
        lookups['user_end'].get((uid, end), 0),
        lookups['global_end'].get(end, 0),
        lookups['hour_end'].get((hour, end), 0),
        lookups['holiday_end'].get((holiday, end), 0),
        lookups['dow_end'].get((dow, end), 0),
        lookups['start_end'].get((start, end), 0),
    ]
    return f + [np.log1p(x) for x in f]


def _df_to_samples(df_raw, df_sugg, lookups, split_name):
    """將某個 split 的行程 DataFrame 轉成 (uid, label, feats) 樣本列表。"""
    valid_users = df_raw['uid_hash'].unique()
    sugg_dict = (df_sugg[df_sugg['uid_hash'].isin(valid_users)]
                        .groupby('uid_hash')['end_latlng']
                        .apply(lambda g: g.drop_duplicates().tolist())
                        .to_dict())
    rows = []
    groups = list(df_raw.groupby('uid_hash', sort=False))
    with tqdm(groups, desc=f'  建立 {split_name} 樣本', unit='user', leave=False) as pbar:
        for uid, grp in pbar:
            cands = sugg_dict.get(uid, [])
            if not cands: continue
            cand_set = set(cands)
            for _, trip in grp.iterrows():
                true_end = trip['end_latlng']
                if true_end not in cand_set: continue
                neg_cands = [c for c in cands if c != true_end]
                n_neg     = min(len(neg_cands), NEG_RATIO)
                sel       = (np.random.choice(len(neg_cands), n_neg, replace=False)
                             if n_neg > 0 else [])
                h, hol, dow, start = (trip['hour_type'], trip['is_holiday'],
                                      trip['dayofweek'], trip['start_latlng'])
                rows.append((uid, 1, _get_feat(lookups, uid, true_end, h, hol, dow, start)))
                for i in sel:
                    rows.append((uid, 0, _get_feat(lookups, uid, neg_cands[i], h, hol, dow, start)))
    return rows


def to_df(rows):
    feats = np.array([r[2] for r in rows], dtype=np.float32)
    df = pd.DataFrame(feats, columns=ALL_FEAT)
    df.insert(0, 'label',    [r[1] for r in rows])
    df.insert(0, 'uid_hash', [r[0] for r in rows])
    return df


def build_samples(df_train_raw, df_val_raw, df_test_raw, df_sugg, lookups):
    print('[3/5] 建立訓練樣本（時序切分，lookups 僅用 train）...')
    np.random.seed(SEED)

    rows_tr   = _df_to_samples(df_train_raw, df_sugg, lookups, 'train')
    rows_val  = _df_to_samples(df_val_raw,   df_sugg, lookups, 'val')
    rows_test = _df_to_samples(df_test_raw,  df_sugg, lookups, 'test')

    df_train = to_df(rows_tr)
    df_val   = to_df(rows_val)
    df_test  = to_df(rows_test)

    for name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
        print(f'  ✓ {name:<6} {len(df):>7,} 樣本  '
              f'pos_rate={df["label"].mean():.3f}')

    # 存到 Drive
    df_train.to_parquet(OUTPUT_DIR / 'samples_train.parquet', index=False)
    df_val.to_parquet(OUTPUT_DIR   / 'samples_val.parquet',   index=False)
    df_test.to_parquet(OUTPUT_DIR  / 'samples_test.parquet',  index=False)
    print('  ✓ 樣本已存至 Drive')
    return df_train, df_val, df_test


df_train, df_val, df_test = build_samples(
    df_train_raw, df_val_raw, df_test_raw, df_sugg, lookups
)

## 6｜訓練 LightGBM LambdaRank

In [ ]:
def _tqdm_lgb_callback(pbar):
    """把 LightGBM iteration 進度接到 tqdm 進度條，並即時顯示 val NDCG@1。"""
    prev = [0]
    def callback(env):
        n = env.iteration + 1
        pbar.update(n - prev[0])
        prev[0] = n
        # evaluation_result_list 順序：[train_ndcg@1, val_ndcg@1, ...]
        # 取最後一個 dataset（val）的 NDCG@1
        if env.evaluation_result_list:
            val_metric = env.evaluation_result_list[-3]  # val NDCG@1
            pbar.set_postfix_str(f'val NDCG@1={val_metric[2]:.4f}')
    callback.order = 10
    return callback


def train_model(df_train: pd.DataFrame, df_val: pd.DataFrame) -> lgb.Booster:
    print('[4/5] 訓練 LightGBM LambdaRank...')

    X_tr  = df_train[ALL_FEAT].values
    y_tr  = df_train['label'].values
    g_tr  = df_train.groupby('uid_hash', sort=False).size().values

    X_val = df_val[ALL_FEAT].values
    y_val = df_val['label'].values
    g_val = df_val.groupby('uid_hash', sort=False).size().values

    ds_train = lgb.Dataset(X_tr,  label=y_tr,  group=g_tr,  feature_name=ALL_FEAT)
    ds_val   = lgb.Dataset(X_val, label=y_val, group=g_val, feature_name=ALL_FEAT,
                           reference=ds_train)  # reference 避免重複 bin 計算

    params = {
        'objective':        'lambdarank',
        'metric':           'ndcg',
        'ndcg_eval_at':     [1, 3, 5],
        'learning_rate':    0.05,
        'num_leaves':       63,
        'min_data_in_leaf': 10,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq':     5,
        'verbosity':        -1,
        'seed':             SEED,
        **LGB_DEVICE_PARAMS,
    }
    print(f'  裝置模式: {"GPU 🟢" if USE_GPU else "CPU 🔵"}')
    print(f'  最大迭代: {N_ROUNDS} rounds，early stopping: {EARLY_STOP} rounds')

    ckpt_path  = OUTPUT_DIR / 'lgbm_checkpoint.txt'
    meta_path  = OUTPUT_DIR / 'train_meta.json'
    init_model = str(ckpt_path) if ckpt_path.exists() else None
    start_iter = 0
    if init_model:
        if meta_path.exists():
            with open(meta_path) as f:
                start_iter = json.load(f).get('n_iter', 0)
        print(f'  發現 checkpoint，從第 {start_iter} 輪繼續...')

    with tqdm(total=start_iter + N_ROUNDS, initial=start_iter,
              desc='  迭代訓練', unit='round') as pbar:
        model = lgb.train(
            params,
            ds_train,
            num_boost_round=N_ROUNDS,
            valid_sets=[ds_train, ds_val],
            valid_names=['train',  'val'],
            callbacks=[
                lgb.log_evaluation(-1),
                lgb.early_stopping(EARLY_STOP, verbose=False),
                _tqdm_lgb_callback(pbar),
            ],
            init_model=init_model,
        )

    best = model.best_iteration
    print(f'\n  ✓ Early stopping 於第 {best} 輪停止')

    # 儲存 checkpoint 到 Drive
    model.save_model(str(ckpt_path))
    with open(OUTPUT_DIR / 'lgbm_ranker.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open(meta_path, 'w') as f:
        json.dump({'n_iter': model.num_trees(), 'best_iteration': best,
                   'feature_names': ALL_FEAT}, f)

    print('  Feature importance (gain):')
    pairs = sorted(zip(ALL_FEAT, model.feature_importance('gain')), key=lambda x: -x[1])
    max_s = pairs[0][1]
    for name, score in pairs:
        bar = '█' * int(score / max_s * 25)
        print(f'    {name:<30} {bar}  {score:.0f}')
    return model


model = train_model(df_train, df_val)

## 7｜評估模型

In [ ]:
def evaluate(model: lgb.Booster, df_test: pd.DataFrame, k_list=TOP_K_LIST) -> dict:
    print('[5/5] 評估模型...')
    df_test = df_test.copy()

    with tqdm(total=2, desc='  評估', unit='step') as pbar:
        pbar.set_postfix_str('推論分數中')
        df_test['score'] = model.predict(df_test[ALL_FEAT].values)
        pbar.update(1)

        pbar.set_postfix_str('計算指標中')
        metrics = {k: {'recall': [], 'mrr': [], 'ndcg': []} for k in k_list}
        for _, grp in df_test.groupby('uid_hash', sort=False):
            labels = grp.sort_values('score', ascending=False)['label'].values
            for k in k_list:
                top = labels[:k]
                metrics[k]['recall'].append(int(top.sum() > 0))
                pos = np.where(top == 1)[0]
                metrics[k]['mrr'].append(1 / (pos[0] + 1) if len(pos) > 0 else 0.0)
                ideal = np.sort(labels)[::-1][:k]
                dcg   = sum(labels[i] / np.log2(i + 2) for i in range(min(k, len(labels))))
                idcg  = sum(ideal[i]  / np.log2(i + 2) for i in range(len(ideal)))
                metrics[k]['ndcg'].append(dcg / idcg if idcg > 0 else 0.0)
        pbar.update(1)

    print(f"\n  {'K':<5} {'Recall@K':>10} {'MRR@K':>10} {'NDCG@K':>10}")
    print(f"  {'-'*38}")
    out = {}
    for k in k_list:
        r = np.mean(metrics[k]['recall'])
        m = np.mean(metrics[k]['mrr'])
        n = np.mean(metrics[k]['ndcg'])
        print(f'  {k:<5} {r:>10.4f} {m:>10.4f} {n:>10.4f}')
        out[f'Recall@{k}'] = round(r, 4)
        out[f'MRR@{k}']    = round(m, 4)
        out[f'NDCG@{k}']   = round(n, 4)

    with open(OUTPUT_DIR / 'eval_results.json', 'w') as f:
        json.dump(out, f, indent=2)
    print(f'\n  ✓ 評估結果已存至 Drive')
    return out


eval_results = evaluate(model, df_test)

## 8｜推薦示範

In [ ]:
def recommend(uid: str, start_latlng: str, hour_type: str,
              is_holiday: str, dayofweek: str,
              model: lgb.Booster, df_sugg: pd.DataFrame,
              lookups: dict, top_k: int = 5) -> list:
    """
    給定用戶與當前叫車情境，回傳 Top-K 下車地址推薦清單。

    Args:
        uid          : 用戶 uid_hash
        start_latlng : 上車經緯度字串，如 '25.05,121.52'
        hour_type    : 時段，如 '早尖峰'
        is_holiday   : '0' 或 '1'
        dayofweek    : '1'~'7'
        top_k        : 回傳前幾名

    Returns:
        list of dict，每筆包含 end_address, end_latlng_pin, score
    """
    cands = (df_sugg[df_sugg['uid_hash'] == uid]
             [['end_latlng', 'end_address', 'end_latlng_pin']]
             .drop_duplicates('end_latlng'))

    if cands.empty:
        # Cold-start：回傳全局最熱門地址
        top_latlngs = sorted(lookups['global_end'].items(), key=lambda x: -x[1])[:top_k]
        fallback = df_sugg[df_sugg['end_latlng'].isin([k for k, _ in top_latlngs])]
        return (fallback[['end_address', 'end_latlng_pin']]
                .drop_duplicates().head(top_k)
                .assign(score=0.0).to_dict('records'))

    rows = []
    for _, row in cands.iterrows():
        rows.append(_get_feat(lookups, uid, row['end_latlng'],
                              hour_type, is_holiday, dayofweek, start_latlng))

    cands = cands.copy()
    cands['score'] = model.predict(np.array(rows, dtype=np.float32))
    return (cands.sort_values('score', ascending=False)
                 .head(top_k)[['end_address', 'end_latlng_pin', 'score']]
                 .to_dict('records'))


# ── 試跑一個真實用戶 ──
sample_uid = df_test['uid_hash'].iloc[0]
recs = recommend(
    uid          = sample_uid,
    start_latlng = '25.05,121.52',
    hour_type    = '早尖峰',
    is_holiday   = '0',
    dayofweek    = '2',
    model        = model,
    df_sugg      = df_sugg,
    lookups      = lookups,
    top_k        = 5,
)

print(f'用戶: {sample_uid[:20]}...')
print(f'{"排名":<4} {"下車地址":<40} {"Score":>8}')
print('-' * 56)
for i, r in enumerate(recs, 1):
    print(f'#{i:<3} {r["end_address"]:<40} {r["score"]:>8.4f}')

---
## 📥 只想重新評估（已有 checkpoint）
如果 runtime 重啟、模型已存在 Drive，執行以下這格就好，不用重新跑全部。

In [ ]:
# ── 從 Drive 載入已訓練模型並重新評估 ────────────────────────────────
with open(OUTPUT_DIR / 'lgbm_ranker.pkl', 'rb') as f:
    model_loaded = pickle.load(f)

df_test_saved = pd.read_parquet(OUTPUT_DIR / 'samples_test.parquet')
evaluate(model_loaded, df_test_saved)